# Predicción de rendimiento con Machine Learning

Este notebook demuestra cómo entrenar un modelo de Random Forest usando datos de la plataforma.

**Requisito**: `scikit-learn` se instala bajo demanda (~15 MB).

In [ ]:
# Instalar scikit-learn on-demand (lazy loading — not pre-bundled)
import micropip
await micropip.install('scikit-learn')
print('scikit-learn installed')

In [ ]:
import nekazari as nkz
import pandas as pd
import numpy as np

# Descargar features de entrenamiento: humedad del suelo y temperatura
# Ajusta device_id a tu sensor real
df = await nkz.timeseries.get_dataframe(
    device_id="120786a0cf364796",
    attributes=["soilMoisture", "airTemperature"],
    start_date="2026-01-01",
    end_date="2026-03-28",
    resolution=1000,
)

print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Preparar features — en un caso real, value_0 y value_1 son las variables
# y necesitarías una columna target (rendimiento real, no disponible aquí).
# Para este tutorial, generamos un target sintético.

feature_cols = [c for c in df.columns if c.startswith('value_')]
df_clean = df[['timestamp'] + feature_cols].dropna()

# Target sintético: combinación lineal + ruido (sustituir por datos reales)
np.random.seed(42)
df_clean = df_clean.copy()
df_clean['yield_synthetic'] = (
    0.3 * df_clean[feature_cols[0]]
    + 0.5 * df_clean[feature_cols[1]]
    + np.random.normal(0, 0.5, len(df_clean))
)

print(f"Features: {feature_cols}")
print(f"Samples: {len(df_clean)}")
df_clean.describe()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

X = df_clean[feature_cols].values
y = df_clean['yield_synthetic'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=50, max_depth=8, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"MAE:  {mean_absolute_error(y_test, y_pred):.3f}")
print(f"R^2:  {r2_score(y_test, y_pred):.3f}")
print(f"\nFeature importances:")
for name, imp in zip(feature_cols, model.feature_importances_):
    print(f"  {name}: {imp:.3f}")

## Notas

- **Target sintético**: Este ejemplo usa datos generados. Para un modelo real, necesitas datos de rendimiento históricos (cosecha kg/ha) asociados a las mismas parcelas/sensores.
- **Features adicionales**: Añade GDD acumulados, precipitación, NDVI (módulo Vegetation Health) para mejorar el modelo.
- **Escalabilidad**: `scikit-learn` en WASM tiene las mismas limitaciones de RAM que tu navegador (~2-4 GB). Para datasets mayores, usa el módulo Intelligence (SSE predictions server-side).